# parameter-wrap-around-tensor — worked example 1: IS-A Parameter survives the isinstance gate

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `parameter-wrap-around-tensor`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Every autograd helper collects trainable state with an `isinstance(_, MiniTensor)` filter. A composition (HAS-A) Parameter that merely stores a tensor as an attribute fails that gate and is silently dropped; only the subclass (IS-A) Parameter passes.

## Worked solution

We build both designs. `WrapParam` stores the tensor in `self.tensor` and is NOT a MiniTensor subclass; `IsAParam(MiniTensor)` inherits, so it IS a MiniTensor. The helper `collect_params(things)` keeps only items passing `isinstance(_, MiniTensor)`. We hand it a mixed list containing one of each. The WrapParam is silently excluded while the IsAParam is kept, demonstrating the bug: composition Parameters become invisible trainable state. We print which designs survived to make the silent drop visible.

In [ ]:
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class WrapParam:  # composition (HAS-A) -- WRONG
    def __init__(self, tensor):
        self.tensor = tensor
        self.requires_grad = True

class IsAParam(MiniTensor):  # subclass (IS-A) -- RIGHT
    def __init__(self, array):
        super().__init__(array, requires_grad=True)

def collect_params(things):
    return [x for x in things if isinstance(x, MiniTensor)]

bag = [WrapParam(np.array([1.0])), IsAParam([2.0])]
survivors = collect_params(bag)
print('survivors:', [type(x).__name__ for x in survivors])